# LLM-as-a-Judge — Top 10 Common Articles Across OPT-125M, OPT-1.3B and OPT-2.7B

This notebook is completely separate from model training.

It:

1. Loads the three saved `*_test_results.csv` files.
2. Finds article IDs shared by all three models.
3. Selects the **same 10 best common articles** using:
   - summary quality: ROUGE-L / ROUGE-2 / SARI when available,
   - aspect quality: Aspect Macro ROUGE-L / Full Target ROUGE-L,
   - conservative source-grounding proxies for generated text.
4. Uses **Qwen2.5-7B-Instruct** as the independent LLM judge.
5. Evaluates only those 10 articles for all three models.
6. Scores:
   - Clarity
   - Usefulness
   - Factual Consistency
   - Grammar
   - Faithfulness
7. Saves short reasons, source evidence quotes, summary omissions, aspect errors, unsupported claims and one improvement suggestion.
8. Produces model-level comparison CSVs.

### Important methodological note

This is a **best-case evaluation subset**, because the 10 articles are deliberately selected from the strongest automatic results. It is useful for qualitative demonstration, but it should not be described as representative of the full 50-article test set.

The source-grounding filter is a conservative **hallucination proxy**, not proof that an output is hallucination-free. The Qwen judge performs the second, semantic check.


In [ ]:
%pip install -q "transformers>=4.46" "accelerate>=1.0" "bitsandbytes>=0.44" "pandas>=2.0" "numpy>=1.26"


In [ ]:
import os
import re
import gc
import glob
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

RESULTS_DIR = Path("results/llm_judge_top10")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

JUDGE_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# 30 judge calls total: 10 articles x 3 models.
TOP_K_ARTICLES = 10

# Standalone judge does not use the target/reference summary.
# It judges Usefulness from the original article itself.
JUDGE_MAX_INPUT_TOKENS = 8192
JUDGE_MAX_NEW_TOKENS = 850
JUDGE_MAX_RETRIES = 2

print("Judge:", JUDGE_MODEL_NAME)
print("Output directory:", RESULTS_DIR)


## 1. Locate the three test-result files

The notebook accepts a few filename variations so you do not need to rename an older result file.

Preferred names:

- `opt-125m_test_results.csv`
- `opt1_3b_test_results.csv`
- `opt2_7b_test_results.csv`


In [ ]:
MODEL_FILE_CANDIDATES = {
    # "OPT-125M": [
    #     "opt125m_test_results.csv",
    # ],
    "OPT-1.3B": [
        "opt1_3b_test_results.csv",
    ],
    "OPT-2.7B": [
        "opt2_7b_test_results.csv",
    ],
}

SEARCH_ROOTS = [
    Path("."),
    Path("/kaggle/working"),
    Path("/kaggle/input"),
]


def find_file(candidates):
    # Fast direct checks first.
    for filename in candidates:
        for root in SEARCH_ROOTS[:2]:
            path = root / filename
            if path.exists():
                return path.resolve()

    # Kaggle datasets may put files inside a dataset subfolder.
    for filename in candidates:
        matches = glob.glob(
            f"/kaggle/input/**/{filename}",
            recursive=True,
        )
        if matches:
            return Path(matches[0]).resolve()

    return None


MODEL_FILES = {
    model_name: find_file(candidates)
    for model_name, candidates in MODEL_FILE_CANDIDATES.items()
}

print("Detected files:")
for model_name, path in MODEL_FILES.items():
    print(f"  {model_name}: {path}")

missing = [
    model_name
    for model_name, path in MODEL_FILES.items()
    if path is None
]

if missing:
    raise FileNotFoundError(
        "Missing test-result file(s) for: "
        + ", ".join(missing)
        + ". Upload the CSV files using one of the supported filenames."
    )


## 2. Load and validate model outputs


In [ ]:
REQUIRED_COLUMNS = {
    "article_id",
    "article",
    "raw_output",
}

OPTIONAL_QUALITY_COLUMNS = [
    "rougeL",
    "rouge2",
    "sari",
    "aspect_macro_rougeL",
    "full_target_rougeL",
]

model_frames = {}

for model_name, path in MODEL_FILES.items():
    frame = pd.read_csv(path)

    missing_required = REQUIRED_COLUMNS - set(frame.columns)
    if missing_required:
        raise ValueError(
            f"{model_name} is missing required columns: "
            f"{sorted(missing_required)}"
        )

    frame["article_id"] = frame["article_id"].astype(str)
    frame["model"] = model_name

    model_frames[model_name] = frame

    print(
        f"{model_name}: {len(frame)} rows | "
        f"{sum(col in frame.columns for col in OPTIONAL_QUALITY_COLUMNS)} "
        f"quality metrics found"
    )

common_ids = set.intersection(
    *[
        set(frame["article_id"])
        for frame in model_frames.values()
    ]
)

print("\nCommon article IDs:", len(common_ids))

if len(common_ids) < TOP_K_ARTICLES:
    raise ValueError(
        f"Only {len(common_ids)} common article IDs were found. "
        f"At least {TOP_K_ARTICLES} are required."
    )


## 3. Conservative automatic selection of the best 10 common articles

The selection is intentionally based on **all three models**, not only one model, so an article must perform reasonably well across the comparison.

### Automatic quality score

For each model/output:

- **Summary score**: ROUGE-L, ROUGE-2 and SARI where available.
- **Aspect score**: Aspect Macro ROUGE-L and Full Target ROUGE-L.
- **Grounding proxy**:
  - meaningful generated words appearing in the source,
  - generated numbers appearing in the source,
  - capitalized names/phrases appearing in the source.

The final article score combines the **mean quality** and **worst-model quality**, preventing one model from dominating selection.

This filter cannot prove absence of hallucination. It is only used to avoid obviously weak/poorly grounded outputs before Qwen evaluation.


In [ ]:
COMMON_STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "if", "while", "of", "to",
    "in", "on", "at", "for", "from", "by", "with", "as", "is", "are",
    "was", "were", "be", "been", "being", "it", "its", "this", "that",
    "these", "those", "he", "she", "they", "them", "his", "her", "their",
    "we", "you", "i", "not", "stated", "summary", "aspects", "event",
    "people", "organization", "date", "time", "location", "causes",
    "effects", "countermeasures", "believed", "cause",
}


def content_tokens(text):
    tokens = re.findall(
        r"[A-Za-z0-9]+(?:['’-][A-Za-z0-9]+)?",
        str(text).lower(),
    )
    return [
        token
        for token in tokens
        if token not in COMMON_STOPWORDS
        and len(token) > 2
    ]


def lexical_source_support(article, output):
    source_tokens = set(content_tokens(article))
    output_tokens = content_tokens(output)

    if not output_tokens:
        return 1.0

    return sum(
        token in source_tokens
        for token in output_tokens
    ) / len(output_tokens)


def numeric_source_support(article, output):
    output_numbers = re.findall(
        r"\b\d+(?:[.,]\d+)?%?\b",
        str(output),
    )

    if not output_numbers:
        return 1.0

    article_text = str(article)

    return sum(
        number in article_text
        for number in output_numbers
    ) / len(output_numbers)


LABEL_PREFIXES = {
    "Summary",
    "Aspects",
    "Event",
    "People",
    "Organization",
    "Date",
    "Time",
    "Location",
    "Causes",
    "Effects",
    "Countermeasures",
    "Not",
    "Believed",
}


def capitalized_phrases(text):
    phrases = re.findall(
        r"\b(?:[A-Z][A-Za-z'’-]+"
        r"(?:\s+[A-Z][A-Za-z'’-]+){0,3})\b",
        str(text),
    )

    return [
        phrase
        for phrase in phrases
        if phrase.split()[0] not in LABEL_PREFIXES
    ]


def capitalized_phrase_support(article, output):
    phrases = capitalized_phrases(output)

    if not phrases:
        return 1.0

    source_lower = str(article).lower()

    return sum(
        phrase.lower() in source_lower
        for phrase in phrases
    ) / len(phrases)


def minmax(series):
    series = pd.to_numeric(
        series,
        errors="coerce",
    )

    if series.notna().sum() == 0:
        return pd.Series(
            np.nan,
            index=series.index,
        )

    minimum = series.min()
    maximum = series.max()

    if np.isclose(minimum, maximum):
        return pd.Series(
            1.0,
            index=series.index,
        )

    return (series - minimum) / (maximum - minimum)


selection_frames = []

for model_name, frame in model_frames.items():
    current = frame[
        frame["article_id"].isin(common_ids)
    ].copy()

    current["lexical_grounding"] = current.apply(
        lambda row: lexical_source_support(
            row["article"],
            row["raw_output"],
        ),
        axis=1,
    )

    current["numeric_grounding"] = current.apply(
        lambda row: numeric_source_support(
            row["article"],
            row["raw_output"],
        ),
        axis=1,
    )

    current["name_grounding"] = current.apply(
        lambda row: capitalized_phrase_support(
            row["article"],
            row["raw_output"],
        ),
        axis=1,
    )

    current["grounding_proxy"] = (
        0.50 * current["lexical_grounding"]
        + 0.25 * current["numeric_grounding"]
        + 0.25 * current["name_grounding"]
    )

    # Very conservative pre-screen for obvious unsupported details.
    current["grounding_pass"] = (
        (current["numeric_grounding"] >= 0.999)
        & (current["name_grounding"] >= 0.75)
        & (current["lexical_grounding"] >= 0.75)
    )

    # Normalize metrics within each model.
    scaled = {}

    for metric in OPTIONAL_QUALITY_COLUMNS:
        if metric in current.columns:
            scaled[metric] = minmax(current[metric])

    summary_parts = [
        scaled[name]
        for name in ["rougeL", "rouge2", "sari"]
        if name in scaled
    ]

    aspect_parts = [
        scaled[name]
        for name in [
            "aspect_macro_rougeL",
            "full_target_rougeL",
        ]
        if name in scaled
    ]

    if not summary_parts:
        raise ValueError(
            f"{model_name}: no summary-quality metric was found."
        )

    if not aspect_parts:
        raise ValueError(
            f"{model_name}: no aspect-quality metric was found."
        )

    current["summary_quality"] = pd.concat(
        summary_parts,
        axis=1,
    ).mean(axis=1)

    current["aspect_quality"] = pd.concat(
        aspect_parts,
        axis=1,
    ).mean(axis=1)

    current["automatic_quality"] = (
        0.45 * current["summary_quality"]
        + 0.40 * current["aspect_quality"]
        + 0.15 * current["grounding_proxy"]
    )

    selection_frames.append(
        current[
            [
                "article_id",
                "model",
                "automatic_quality",
                "summary_quality",
                "aspect_quality",
                "grounding_proxy",
                "grounding_pass",
            ]
        ]
    )

selection_long = pd.concat(
    selection_frames,
    ignore_index=True,
)

article_selection = (
    selection_long
    .groupby("article_id")
    .agg(
        mean_quality=("automatic_quality", "mean"),
        worst_model_quality=("automatic_quality", "min"),
        mean_summary_quality=("summary_quality", "mean"),
        mean_aspect_quality=("aspect_quality", "mean"),
        mean_grounding=("grounding_proxy", "mean"),
        all_models_grounding_pass=("grounding_pass", "all"),
        model_count=("model", "nunique"),
    )
    .reset_index()
)

article_selection["selection_score"] = (
    0.70 * article_selection["mean_quality"]
    + 0.30 * article_selection["worst_model_quality"]
)

# Prefer only articles whose three outputs pass the conservative grounding proxy.
eligible = article_selection[
    article_selection["all_models_grounding_pass"]
].copy()

if len(eligible) >= TOP_K_ARTICLES:
    selected_top10 = (
        eligible
        .sort_values(
            [
                "selection_score",
                "mean_grounding",
            ],
            ascending=False,
        )
        .head(TOP_K_ARTICLES)
        .copy()
    )
    selected_top10["selection_mode"] = (
        "All 3 models passed grounding pre-screen"
    )
else:
    print(
        f"Only {len(eligible)} articles passed the strict grounding screen. "
        "Filling the remaining positions using the highest grounding/quality scores."
    )

    selected_top10 = (
        article_selection
        .sort_values(
            [
                "all_models_grounding_pass",
                "mean_grounding",
                "selection_score",
            ],
            ascending=False,
        )
        .head(TOP_K_ARTICLES)
        .copy()
    )

    selected_top10["selection_mode"] = np.where(
        selected_top10["all_models_grounding_pass"],
        "All 3 models passed grounding pre-screen",
        "Fallback: strongest remaining grounding/quality",
    )

selected_top10.insert(
    0,
    "rank",
    range(1, len(selected_top10) + 1),
)

SELECTED_ARTICLE_IDS = selected_top10[
    "article_id"
].tolist()

selected_path = RESULTS_DIR / "selected_top10_articles.csv"
selected_top10.to_csv(
    selected_path,
    index=False,
)

print("\nSelected 10 article IDs:")
display(selected_top10)

print("Saved:", selected_path)


## 4. Build the 30-output evaluation set

The same 10 article IDs are extracted from every model, creating:

**10 articles × 3 models = 30 LLM-judge evaluations**


In [ ]:
evaluation_rows = []

for model_name, frame in model_frames.items():
    subset = (
        frame[
            frame["article_id"].isin(SELECTED_ARTICLE_IDS)
        ]
        .copy()
    )

    subset["selection_rank"] = subset[
        "article_id"
    ].map({
        article_id: rank
        for rank, article_id in enumerate(
            SELECTED_ARTICLE_IDS,
            start=1,
        )
    })

    subset = subset.sort_values(
        "selection_rank"
    )

    for _, row in subset.iterrows():
        evaluation_rows.append({
            "selection_rank": row["selection_rank"],
            "article_id": row["article_id"],
            "model": model_name,
            "article": row["article"],
            "generated_output": row["raw_output"],
            "generated_summary": row.get(
                "generated_summary",
                "",
            ),
            "generated_aspects": row.get(
                "generated_aspects",
                "",
            ),
            "rougeL": row.get("rougeL", np.nan),
            "rouge2": row.get("rouge2", np.nan),
            "sari": row.get("sari", np.nan),
            "aspect_macro_rougeL": row.get(
                "aspect_macro_rougeL",
                np.nan,
            ),
            "full_target_rougeL": row.get(
                "full_target_rougeL",
                np.nan,
            ),
        })

evaluation_df = pd.DataFrame(
    evaluation_rows
).sort_values(
    ["selection_rank", "model"]
).reset_index(drop=True)

assert len(evaluation_df) == TOP_K_ARTICLES * 3

print("LLM-judge outputs to evaluate:", len(evaluation_df))
display(
    evaluation_df[
        [
            "selection_rank",
            "article_id",
            "model",
            "rougeL",
            "aspect_macro_rougeL",
        ]
    ]
)


## 5. Qwen LLM-as-a-Judge rubric

The judge does **not** receive the dataset/reference summary.

This prevents Usefulness from becoming another ROUGE-like comparison.

- **Usefulness**: important coverage according to the original article.
- **Factual Consistency**: contradictions with the original article.
- **Faithfulness**: unsupported claims actually made by the model.
- Missing information can reduce Usefulness but does not automatically reduce Faithfulness.


In [ ]:
JUDGE_RUBRICS = {
    "clarity": {
        1: "Hard to understand overall; major wording or organization problems prevent comprehension.",
        2: "Several parts are difficult to understand and require rereading.",
        3: "Generally understandable, but one notable confusing section or several awkward/overly complex passages remain.",
        4: "Clear and easy to follow with at most one minor awkward or unnecessarily complex passage.",
        5: "Very clear, concise and easy to follow throughout with no meaningful clarity problem.",
    },
    "usefulness": {
        1: "Misses or badly misrepresents the central news event.",
        2: "Only partially communicates the main story; several major facts or a critical outcome are missing.",
        3: "Covers the central event but omits one major fact or multiple useful supporting facts.",
        4: "Covers the central event and nearly all important information; at most one minor useful detail is omitted.",
        5: "Efficiently covers the central event plus the essential outcomes, consequences or responses.",
    },
    "factual_consistency": {
        1: "The central story is factually wrong or there are many major contradictions.",
        2: "Contains multiple major factual contradictions that materially distort the article.",
        3: "Contains one major factual error or multiple minor factual errors.",
        4: "Factually consistent overall with at most one minor imprecision that does not change the main meaning.",
        5: "No factual contradiction or meaningful factual error is detected.",
    },
    "grammar": {
        1: "Frequent serious grammatical errors make the output difficult to understand.",
        2: "Several grammatical or sentence-formation errors noticeably disrupt reading.",
        3: "Understandable overall but contains multiple minor issues or one substantial grammar problem.",
        4: "Grammatically strong with at most one minor awkward or incorrect construction.",
        5: "Grammatically correct and natural throughout.",
    },
    "faithfulness": {
        1: "The output is largely unsupported or invented and substantially misrepresents the source.",
        2: "Contains multiple major unsupported claims or inferences.",
        3: "Contains one major unsupported claim or multiple minor unsupported/over-inferred claims.",
        4: "Nearly all claims are source-supported, with at most one minor overstatement or weak inference.",
        5: "All meaningful generated claims are supported by the source; no unsupported claim is detected.",
    },
}

print(json.dumps(JUDGE_RUBRICS, indent=2))


## 6. Load Qwen2.5-7B-Instruct in 4-bit

Only the judge model is loaded. None of the OPT models are loaded or retrained.


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "A CUDA GPU is recommended for Qwen2.5-7B-Instruct 4-bit evaluation."
    )

GPU_CAPABILITY = torch.cuda.get_device_capability(0)
USE_BF16 = (
    GPU_CAPABILITY[0] >= 8
    and torch.cuda.is_bf16_supported()
)
COMPUTE_DTYPE = (
    torch.bfloat16
    if USE_BF16
    else torch.float16
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

judge_tokenizer = AutoTokenizer.from_pretrained(
    JUDGE_MODEL_NAME,
    use_fast=True,
)

if judge_tokenizer.pad_token is None:
    judge_tokenizer.pad_token = judge_tokenizer.eos_token

judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=COMPUTE_DTYPE,
)

judge_model.eval()
judge_model.config.use_cache = True

JUDGE_DEVICE = next(
    parameter.device
    for parameter in judge_model.parameters()
    if parameter.device.type != "meta"
)

print("Judge device:", JUDGE_DEVICE)


In [ ]:
JUDGE_SYSTEM_PROMPT = """
You are a strict and consistent evaluator of a News Simplification Assistant.

Use ONLY the original article supplied by the user as factual authority.
Do not use outside knowledge.
Do not compare the generated output with a hidden/reference summary.
Do not provide hidden chain-of-thought.
Return concise JSON only.
""".strip()


def rubric_text():
    sections = []

    for criterion, levels in JUDGE_RUBRICS.items():
        lines = [
            f"{criterion.replace('_', ' ').title()}:"
        ]
        for score in range(1, 6):
            lines.append(
                f"  {score}: {levels[score]}"
            )
        sections.append("\n".join(lines))

    return "\n\n".join(sections)


def build_judge_prompt(article, generated_output):
    schema = {
        "clarity": {
            "score": 5,
            "reason": "short decisive reason",
            "evidence_quote": ""
        },
        "usefulness": {
            "score": 5,
            "reason": "short decisive reason",
            "evidence_quote": "short exact source quote if useful"
        },
        "factual_consistency": {
            "score": 5,
            "reason": "short decisive reason",
            "evidence_quote": "short exact source quote if relevant"
        },
        "grammar": {
            "score": 5,
            "reason": "short decisive reason",
            "evidence_quote": ""
        },
        "faithfulness": {
            "score": 5,
            "reason": "short decisive reason",
            "evidence_quote": "short exact source quote if relevant"
        },
        "summary_omissions": [
            {
                "information": "important fact missing from Summary only",
                "evidence_quote": "short exact source quote",
                "reason": "why this omission matters"
            }
        ],
        "aspect_errors": [
            {
                "field": "Effects",
                "generated_value": "Not stated",
                "evidence_quote": "short exact source quote",
                "reason": "why the structured aspect is incorrect"
            }
        ],
        "unsupported_claims": [
            {
                "generated_claim": "claim made by the generated output",
                "reason": "why the source does not support it"
            }
        ],
        "main_issue": "single main weakness, or None",
        "improvement_suggestion": "one concise future improvement"
    }

    return f"""
Evaluate the following generated output.

ORIGINAL ARTICLE:
{article}

GENERATED OUTPUT:
{generated_output}

SCORING RUBRIC:
{rubric_text()}

IMPORTANT RULES:
- Score every criterion from 1 to 5 using the rubric above.
- Judge Usefulness from the ORIGINAL ARTICLE itself.
- A generated summary may choose different important facts and still be useful.
- Missing information may reduce Usefulness.
- Missing information alone does NOT reduce Faithfulness.
- Faithfulness only concerns unsupported/invented/over-inferred claims actually made.
- Factual Consistency concerns contradictions or incorrect facts.
- summary_omissions concerns the Summary section only.
- aspect_errors concerns the structured aspect fields only.
- Do not call something an aspect error if that aspect already correctly contains the fact.
- Allowed aspect fields:
  Event, People, Organization, Date & Time, Location, Causes, Effects, Countermeasures.
- Do not number source sentences.
- Evidence must be a SHORT VERBATIM quote from the original article.
- If no evidence quote is necessary, use "".
- If a diagnostic list has no valid item, return [].
- Keep reasons concise.
- Return valid JSON only. No Markdown fences.

Return exactly this top-level structure:
{json.dumps(schema, ensure_ascii=False, indent=2)}
""".strip()


def extract_json_object(text):
    cleaned = str(text).strip()
    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.I,
    )
    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned,
    )

    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start == -1 or end == -1 or end <= start:
        raise ValueError(
            "Judge response does not contain a JSON object."
        )

    return json.loads(
        cleaned[start:end + 1]
    )


def normalize_score(value):
    try:
        score = int(round(float(value)))
    except Exception:
        raise ValueError(
            f"Invalid score: {value}"
        )

    if score not in {1, 2, 3, 4, 5}:
        raise ValueError(
            f"Score outside 1-5: {score}"
        )

    return score


def normalize_text(value):
    return re.sub(
        r"\s+",
        " ",
        str(value or ""),
    ).strip()


def quote_is_valid(quote, article):
    quote = normalize_text(quote)
    if not quote:
        return True

    return (
        quote.casefold()
        in normalize_text(article).casefold()
    )


ALLOWED_ASPECT_FIELDS = {
    "Event",
    "People",
    "Organization",
    "Date & Time",
    "Location",
    "Causes",
    "Effects",
    "Countermeasures",
}


def validate_payload(payload, article):
    result = {}

    for criterion in JUDGE_RUBRICS:
        item = payload.get(
            criterion,
            {},
        )

        if not isinstance(item, dict):
            raise ValueError(
                f"{criterion} is not an object."
            )

        quote = normalize_text(
            item.get(
                "evidence_quote",
                "",
            )
        )

        result[criterion] = {
            "score": normalize_score(
                item.get("score")
            ),
            "reason": normalize_text(
                item.get("reason", "")
            ),
            "evidence_quote": quote,
            "evidence_quote_valid": quote_is_valid(
                quote,
                article,
            ),
        }

    summary_omissions = []
    for item in payload.get(
        "summary_omissions",
        [],
    ) or []:
        if not isinstance(item, dict):
            continue

        quote = normalize_text(
            item.get(
                "evidence_quote",
                "",
            )
        )

        summary_omissions.append({
            "information": normalize_text(
                item.get("information", "")
            ),
            "evidence_quote": quote,
            "evidence_quote_valid": quote_is_valid(
                quote,
                article,
            ),
            "reason": normalize_text(
                item.get("reason", "")
            ),
        })

    aspect_errors = []
    for item in payload.get(
        "aspect_errors",
        [],
    ) or []:
        if not isinstance(item, dict):
            continue

        field = normalize_text(
            item.get("field", "")
        )

        if field not in ALLOWED_ASPECT_FIELDS:
            continue

        quote = normalize_text(
            item.get(
                "evidence_quote",
                "",
            )
        )

        aspect_errors.append({
            "field": field,
            "generated_value": normalize_text(
                item.get(
                    "generated_value",
                    "",
                )
            ),
            "evidence_quote": quote,
            "evidence_quote_valid": quote_is_valid(
                quote,
                article,
            ),
            "reason": normalize_text(
                item.get("reason", "")
            ),
        })

    unsupported_claims = []
    for item in payload.get(
        "unsupported_claims",
        [],
    ) or []:
        if not isinstance(item, dict):
            continue

        unsupported_claims.append({
            "generated_claim": normalize_text(
                item.get(
                    "generated_claim",
                    "",
                )
            ),
            "reason": normalize_text(
                item.get("reason", "")
            ),
        })

    result["summary_omissions"] = summary_omissions
    result["aspect_errors"] = aspect_errors
    result["unsupported_claims"] = unsupported_claims
    result["main_issue"] = normalize_text(
        payload.get(
            "main_issue",
            "",
        )
    )
    result["improvement_suggestion"] = normalize_text(
        payload.get(
            "improvement_suggestion",
            "",
        )
    )

    return result


def judge_one(article, generated_output):
    user_prompt = build_judge_prompt(
        article,
        generated_output,
    )

    messages = [
        {
            "role": "system",
            "content": JUDGE_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    chat_text = judge_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    encoded = judge_tokenizer(
        chat_text,
        return_tensors="pt",
        add_special_tokens=False,
        truncation=False,
    )

    input_length = int(
        encoded["input_ids"].shape[1]
    )

    if input_length > JUDGE_MAX_INPUT_TOKENS:
        raise ValueError(
            f"Judge prompt contains {input_length} tokens, "
            f"above the {JUDGE_MAX_INPUT_TOKENS} limit."
        )

    encoded = {
        key: value.to(JUDGE_DEVICE)
        for key, value in encoded.items()
    }

    last_error = ""

    for attempt in range(
        1,
        JUDGE_MAX_RETRIES + 1,
    ):
        start = time.perf_counter()

        with torch.inference_mode():
            output_ids = judge_model.generate(
                **encoded,
                max_new_tokens=JUDGE_MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=judge_tokenizer.pad_token_id,
                eos_token_id=judge_tokenizer.eos_token_id,
            )

        elapsed = time.perf_counter() - start

        generated_ids = output_ids[
            0
        ][input_length:]

        raw_output = judge_tokenizer.decode(
            generated_ids,
            skip_special_tokens=True,
        ).strip()

        try:
            parsed = extract_json_object(
                raw_output
            )

            validated = validate_payload(
                parsed,
                article,
            )

            return {
                "parse_ok": True,
                "attempts": attempt,
                "generation_time_sec": elapsed,
                "raw_output": raw_output,
                "payload": validated,
                "error": "",
            }

        except Exception as exc:
            last_error = str(exc)

    return {
        "parse_ok": False,
        "attempts": JUDGE_MAX_RETRIES,
        "generation_time_sec": elapsed,
        "raw_output": raw_output,
        "payload": None,
        "error": last_error,
    }


## 7. Evaluate the 30 selected outputs


In [ ]:
judge_rows = []

for position, (_, row) in enumerate(
    evaluation_df.iterrows(),
    start=1,
):
    result = judge_one(
        article=row["article"],
        generated_output=row["generated_output"],
    )

    record = {
        "selection_rank": row["selection_rank"],
        "article_id": row["article_id"],
        "model": row["model"],
        "judge_model": JUDGE_MODEL_NAME,
        "judge_parse_ok": result["parse_ok"],
        "judge_attempts": result["attempts"],
        "judge_generation_time_sec": result[
            "generation_time_sec"
        ],
        "judge_error": result["error"],
        "judge_raw_output": result["raw_output"],
    }

    payload = result.get("payload")

    if payload:
        scores = []

        for criterion in JUDGE_RUBRICS:
            item = payload[criterion]

            scores.append(
                item["score"]
            )

            record[
                f"llm_{criterion}_score"
            ] = item["score"]

            record[
                f"llm_{criterion}_reason"
            ] = item["reason"]

            record[
                f"llm_{criterion}_evidence_quote"
            ] = item["evidence_quote"]

            record[
                f"llm_{criterion}_evidence_quote_valid"
            ] = item[
                "evidence_quote_valid"
            ]

        record[
            "llm_overall_mean_score"
        ] = float(np.mean(scores))

        record[
            "llm_summary_omissions"
        ] = json.dumps(
            payload["summary_omissions"],
            ensure_ascii=False,
        )

        record[
            "llm_aspect_errors"
        ] = json.dumps(
            payload["aspect_errors"],
            ensure_ascii=False,
        )

        record[
            "llm_unsupported_claims"
        ] = json.dumps(
            payload["unsupported_claims"],
            ensure_ascii=False,
        )

        record[
            "llm_main_issue"
        ] = payload["main_issue"]

        record[
            "llm_improvement_suggestion"
        ] = payload[
            "improvement_suggestion"
        ]

    judge_rows.append(record)

    if (
        position % 5 == 0
        or position == len(evaluation_df)
    ):
        print(
            f"Judged {position}/{len(evaluation_df)} outputs"
        )

llm_results_df = pd.DataFrame(
    judge_rows
)

display(
    llm_results_df[
        [
            "selection_rank",
            "article_id",
            "model",
            "llm_clarity_score",
            "llm_usefulness_score",
            "llm_factual_consistency_score",
            "llm_grammar_score",
            "llm_faithfulness_score",
            "llm_overall_mean_score",
        ]
    ]
)


## 8. Save results and model comparison


In [ ]:
per_output_path = (
    RESULTS_DIR
    / "llm_judge_top10_all_models.csv"
)

llm_results_df.to_csv(
    per_output_path,
    index=False,
)

score_columns = [
    f"llm_{criterion}_score"
    for criterion in JUDGE_RUBRICS
]

successful = llm_results_df[
    llm_results_df["judge_parse_ok"] == True
].copy()

summary_rows = []

for model_name, group in successful.groupby(
    "model"
):
    row = {
        "model": model_name,
        "evaluated_outputs": len(group),
        "parse_success_rate": (
            len(group)
            / TOP_K_ARTICLES
        ),
        "mean_judge_time_sec": group[
            "judge_generation_time_sec"
        ].mean(),
    }

    for criterion in JUDGE_RUBRICS:
        row[
            f"mean_{criterion}"
        ] = pd.to_numeric(
            group[
                f"llm_{criterion}_score"
            ],
            errors="coerce",
        ).mean()

    row[
        "mean_overall"
    ] = pd.to_numeric(
        group[
            "llm_overall_mean_score"
        ],
        errors="coerce",
    ).mean()

    summary_rows.append(row)

model_summary_df = pd.DataFrame(
    summary_rows
).sort_values(
    "mean_overall",
    ascending=False,
)

model_summary_path = (
    RESULTS_DIR
    / "llm_judge_top10_model_summary.csv"
)

model_summary_df.to_csv(
    model_summary_path,
    index=False,
)

# Paired article-level comparison table.
paired = successful.pivot(
    index="article_id",
    columns="model",
    values="llm_overall_mean_score",
).reset_index()

paired_path = (
    RESULTS_DIR
    / "llm_judge_top10_paired_scores.csv"
)

paired.to_csv(
    paired_path,
    index=False,
)

print("\nModel comparison:")
display(model_summary_df)

print("\nSaved:")
print(per_output_path)
print(model_summary_path)
print(paired_path)
print(RESULTS_DIR / "selected_top10_articles.csv")


## Interpretation

Use `llm_judge_top10_model_summary.csv` for the final three-model comparison.

Remember to describe this as a **top-10 best-case qualitative evaluation**, because the subset was selected using automatic quality metrics. Your full-test automatic metrics should remain the primary unbiased model comparison.
